# AI TDD Orchestrator - Kaggle GPU Server

**Alternative to Google Colab** - Kaggle gives 30 free GPU hours per week.

### Setup:
1. Import this notebook to Kaggle
2. Turn ON the GPU accelerator (Settings > Accelerator > GPU P100)
3. Run all cells
4. Add the URL as `KAGGLE_OLLAMA_URL` secret in GitHub

## Cell 1: Install Dependencies and Ollama

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd pciutils > /dev/null 2>&1
print('[OK] Dependencies installed')

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'WARNING: No GPU. Enable GPU in Settings.'

!curl -fsSL https://ollama.com/install.sh | sh
print('[OK] Ollama installed')

## Cell 2: Start Ollama and Pull Model

In [ ]:
import subprocess, time, os

env = os.environ.copy()
env['OLLAMA_HOST'] = '0.0.0.0'
process = subprocess.Popen(
    ['/usr/local/bin/ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env,
)
time.sleep(5)
print('[OK] Ollama server started')

# P100 has 16GB VRAM - 7b fits perfectly
!ollama pull qwen2.5-coder:7b

import requests
try:
    tags = requests.get('http://localhost:11434/api/tags').json()
    names = [m['name'] for m in tags.get('models', [])]
    print(f'[OK] Models loaded: {names}')
except Exception as e:
    print(f'[ERROR] {e}')

## Cell 3: Create Public Tunnel

Paste your ngrok token below (free at https://dashboard.ngrok.com)

In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok

# ==================================================
# PASTE YOUR NGROK AUTH TOKEN HERE:
NGROK_TOKEN = ''  # <-- from https://dashboard.ngrok.com
# ==================================================

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    print('[WARNING] Set NGROK_TOKEN above first')

tunnel = ngrok.connect(11434)
public_url = tunnel.public_url

print()
print('=' * 60)
print('  KAGGLE GPU SERVER IS LIVE!')
print('=' * 60)
print(f'  URL: {public_url}')
print(f'  GitHub Secret Name:  KAGGLE_OLLAMA_URL')
print(f'  GitHub Secret Value: {public_url}')
print('=' * 60)

## Cell 4: Keep Alive

In [ ]:
import time, requests

print(f'Kaggle GPU Server: {public_url}')
print('Press Stop when done.')
print()

counter = 0
while True:
    counter += 1
    try:
        requests.get('http://localhost:11434/api/tags', timeout=5)
    except Exception:
        pass
    time.sleep(60)
    if counter % 5 == 0:
        print(f'Still alive ({counter} min) | URL: {public_url}')